In [11]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_final.csv.gz", parse_dates=["Date", "Store_Open_Date"])

# Cifras generales
print("Ventas (filas):", len(df))
print("Unidades vendidas:", df["Units"].sum())
print("Ingresos totales:", round(df["Total_Revenue"].sum(), 2))
print("Beneficio total:", round(df["Profit"].sum(), 2))
print("Margen global (%):", round(df["Profit"].sum() / df["Total_Revenue"].sum() * 100, 2))

# Por categoría de producto
por_categoria = df.groupby("Product_Category").agg(
    ingresos=("Total_Revenue", "sum"),
    beneficio=("Profit", "sum"),
    unidades=("Units", "sum"),
).round(2)
por_categoria["margen_pct"] = (por_categoria["beneficio"] / por_categoria["ingresos"] * 100).round(2)
print(por_categoria.sort_values("beneficio", ascending=False))

Ventas (filas): 829262
Unidades vendidas: 1090565
Ingresos totales: 14444572.35
Beneficio total: 4014029.0
Margen global (%): 27.79
                     ingresos  beneficio  unidades  margen_pct
Product_Category                                              
Toys               5093241.00  1079527.0    267200       21.20
Electronics        2246771.25  1001437.0    134075       44.57
Art & Crafts       2705364.26   753354.0    325574       27.85
Games              2226836.27   673993.0    194673       30.27
Sports & Outdoors  2172359.57   505718.0    169043       23.28


In [2]:
# Por tipo de ubicación
por_ubicacion = df.groupby("Store_Location").agg(
    tiendas=("Store_ID", "nunique"),
    ingresos=("Total_Revenue", "sum"),
    beneficio=("Profit", "sum"),
).round(2)
por_ubicacion["beneficio_por_tienda"] = (por_ubicacion["beneficio"] / por_ubicacion["tiendas"]).round(2)
print(por_ubicacion.sort_values("beneficio", ascending=False))

# Top 5 ciudades por beneficio
print()
print(df.groupby("Store_City")["Profit"].sum().round(2).sort_values(ascending=False).head(5))

# Beneficio por mes
print()
por_mes = df.groupby(["Year", "Month"])["Profit"].sum().round(2)
print(por_mes)

                tiendas    ingresos  beneficio  beneficio_por_tienda
Store_Location                                                      
Downtown             29  8219596.27  2248728.0              77542.34
Commercial           12  3279139.52   926864.0              77238.67
Residential           6  1656113.98   460388.0              76731.33
Airport               3  1289722.58   378049.0             126016.33

Store_City
Cuidad de Mexico    465558.0
Guadalajara         368930.0
Monterrey           346729.0
Hermosillo          263608.0
Guanajuato          235047.0
Name: Profit, dtype: float64

Year  Month
2022  1        167126.0
      2        161861.0
      3        173992.0
      4        190099.0
      5        186894.0
      6        189815.0
      7        176922.0
      8        158931.0
      9        166397.0
      10       178799.0
      11       192873.0
      12       246078.0
2023  1        205074.0
      2        189314.0
      3        231909.0
      4        215096.0
   

In [8]:
from scipy import stats

# Comprobar el error
df = pd.read_csv("../data/processed/dataset_final.csv.gz", parse_dates=["Date", "Store_Open_Date"])
print(df.shape)
print("Cuidad de Mexico aún existe:", (df["Store_City"] == "Cuidad de Mexico").any())

# Ingresos totales por día
diario = df.groupby(["Date", "Is_Weekend"])["Total_Revenue"].sum().reset_index()

fin_semana = diario[diario["Is_Weekend"]]["Total_Revenue"]
entre_semana = diario[~diario["Is_Weekend"]]["Total_Revenue"]

print("\nDías de fin de semana:", len(fin_semana), "| media diaria:", round(fin_semana.mean(), 2))
print("Días entre semana:", len(entre_semana), "| media diaria:", round(entre_semana.mean(), 2))

t, p = stats.ttest_ind(fin_semana, entre_semana, equal_var=False)
print("\nEstadístico t:", round(t, 3))
print("p-valor:", p)

(829262, 24)
Cuidad de Mexico aún existe: False

Días de fin de semana: 183 | media diaria: 25190.92
Días entre semana: 455 | media diaria: 21614.58

Estadístico t: 4.756
p-valor: 3.2096073492992864e-06


In [9]:
from scipy import stats

# Releer el dataset 
df = pd.read_csv("../data/processed/dataset_final.csv.gz", parse_dates=["Date", "Store_Open_Date"])

# Una fila por tienda
tienda = df.groupby(["Store_ID", "Store_City", "Store_Location", "Store_Open_Date"]).agg(
    beneficio=("Profit", "sum"),
    ingresos=("Total_Revenue", "sum"),
).reset_index()
tienda["antiguedad"] = (pd.Timestamp("2023-09-30") - tienda["Store_Open_Date"]).dt.days / 365.25

print("Tiendas:", len(tienda))

# 1) Aeropuerto vs resto
aeropuerto = tienda[tienda["Store_Location"] == "Airport"]["beneficio"]
resto = tienda[tienda["Store_Location"] != "Airport"]["beneficio"]
print("\nTiendas aeropuerto:", len(aeropuerto), "| beneficio medio:", round(aeropuerto.mean(), 0))
print("Resto de tiendas:", len(resto), "| beneficio medio:", round(resto.mean(), 0))
t, p = stats.ttest_ind(aeropuerto, resto, equal_var=False)
print("Test t de Welch -> t:", round(t, 3), "| p-valor:", round(p, 4))

# 2) Antigüedad vs beneficio
r, p2 = stats.pearsonr(tienda["antiguedad"], tienda["beneficio"])
print("\nCorrelación antigüedad-beneficio -> r:", round(r, 3), "| p-valor:", round(p2, 4))

Tiendas: 50

Tiendas aeropuerto: 3 | beneficio medio: 126016.0
Resto de tiendas: 47 | beneficio medio: 77361.0
Test t de Welch -> t: 2.009 | p-valor: 0.1804

Correlación antigüedad-beneficio -> r: -0.064 | p-valor: 0.6569


In [10]:
# Una fila por producto
producto = df.groupby(["Product_ID", "Product_Name", "Product_Category", "Product_Price"]).agg(
    unidades=("Units", "sum"),
    beneficio=("Profit", "sum"),
).reset_index()

print("Productos:", len(producto))

# Correlación precio vs unidades vendidas 
rho, p = stats.spearmanr(producto["Product_Price"], producto["unidades"])
print("Precio vs unidades -> rho:", round(rho, 3), "| p-valor:", round(p, 4))

# Los 5 más vendidos y los 5 menos vendidos
print()
print(producto.sort_values("unidades", ascending=False)[["Product_Name", "Product_Category", "Product_Price", "unidades"]].head(5))
print()
print(producto.sort_values("unidades")[["Product_Name", "Product_Category", "Product_Price", "unidades"]].head(5))

Productos: 35
Precio vs unidades -> rho: -0.218 | p-valor: 0.2091

       Product_Name Product_Category  Product_Price  unidades
5         Colorbuds      Electronics          14.99    104368
24      PlayDoh Can     Art & Crafts           2.99    103128
2   Barrel O' Slime     Art & Crafts           3.99     91663
7     Deck Of Cards            Games           6.99     84034
18       Magic Sand     Art & Crafts          15.99     60598

            Product_Name   Product_Category  Product_Price  unidades
19  Mini Basketball Hoop  Sports & Outdoors          24.99      2647
34         Uno Card Game              Games           7.99      2710
21              Monopoly              Games          19.99      3385
3       Chutes & Ladders              Games          12.99      3829
27              Playfoam       Art & Crafts          10.99      4158
